# HPO LunarLander

Optuna hyperparameter optimization for `TunedTrainer` on LunarLander.

The happy path on Colab is labeled HP1 to HP4.

Hardware: L4 GPU is the intended runtime.

## Setup

### Runtime setup

Set up the runtime by running exactly one code cell in this section.

#### Colab

In [ ]:
# HP1
!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
!pip install -r hpo/requirements.txt

from pathlib import Path
import sys

from google.colab import drive

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")

study_dir = Path("/content/drive/MyDrive/rl_lab/hpo")
run_dir = Path("/content/rl_lab/hpo/runs")
drive.mount("/content/drive")
study_dir.mkdir(parents=True, exist_ok=True)
run_dir.mkdir(parents=True, exist_ok=True)

HPO_STUDY_DIR = study_dir
HPO_RUN_DIR = run_dir

#### Local

In [ ]:
# Local setup
from pathlib import Path
import sys

sys.path.insert(0, "../../dqn/src")
sys.path.insert(0, "../src")

HPO_STUDY_DIR = Path("../runs")
HPO_RUN_DIR = Path("../runs/episodes")
HPO_STUDY_DIR.mkdir(parents=True, exist_ok=True)
HPO_RUN_DIR.mkdir(parents=True, exist_ok=True)

## Imports

In [ ]:
# HP2
from IPython.display import clear_output, display
import optuna
import torch
from optuna.trial import TrialState
from optuna.visualization import plot_optimization_history

from dqn.training import TrainingConfig
from dqn.tuned_training import TuningConfig
from hpo.evaluation.pruning import PruningConfig
from hpo.lunar_lander.objective import create_objective

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Optimization

### Define SearchSpace

In [ ]:
# HP3
class SearchSpace:
    def replay_memory_capacity(self, trial):
        return 50_000

    def training_config(self, trial, num_episodes):
        return TrainingConfig(
            num_episodes=num_episodes,
            batch_size=128,
            gamma=0.99,
            eps_start=trial.suggest_float("eps_start", 0.5, 1.0),
            eps_end=0.01,
            eps_decay=trial.suggest_int("eps_decay", 10_000, 50_000, log=True),
            tau=0.005,
            learning_rate=3e-4,
        )

    def tuning_config(self, trial, *, output_dir):
        log_path = None
        if output_dir is not None:
            trial_dir = output_dir / f"trial_{trial.number:04d}"
            log_path = trial_dir / "episodes.csv"

        return TuningConfig(
            learning_starts=5_000,
            optimize_every=4,
            double_dqn=False,
            save_best_checkpoint=False,
            checkpoint_min_score=0.0,
            checkpoint_min_score_delta=0.0,
            log_path=log_path,
        )


### Optimize

In [ ]:
# HP4
pruning_config = PruningConfig(
    start_episode=300,
    min_score=100.0,
)

objective = create_objective(
    search_space=SearchSpace(),
    num_episodes=500,
    score_window=50,
    seed=42,
    output_dir=HPO_RUN_DIR,
    device=device,
    pruning_config=pruning_config,
)

study = optuna.create_study(
    study_name="lunar_lander_dqn",
    direction="maximize",
    storage=f"sqlite:///{HPO_STUDY_DIR / 'lunar_lander_dqn.db'}",
    load_if_exists=True,
)

target_trials = 20
complete_trials = lambda: len(study.get_trials(states=(TrialState.COMPLETE,)))
pruned_trials = lambda: len(study.get_trials(states=(TrialState.PRUNED,)))
finished_trials = lambda: complete_trials() + pruned_trials()

while finished_trials() < target_trials:
    study.optimize(objective, n_trials=1)

    # Plot and some text output
    clear_output(wait=True)

    print(f"Target trials: {target_trials}")
    print(f"Finished trials: {finished_trials()}")
    print(f"Complete trials: {complete_trials()}")
    print(f"Pruned trials: {pruned_trials()}")
    print(
        "Episodes saved by pruning:",
        sum(trial.user_attrs.get("episodes_saved_by_pruning", 0) for trial in study.trials),
    )

    if complete_trials() > 0:
        best_trial = study.best_trial
        print(f"Best mean return: {best_trial.value:.1f}")
        print(
            "Best episode window:",
            best_trial.user_attrs["best_window_start_episode"],
            "-",
            best_trial.user_attrs["best_window_end_episode"],
        )
        print("Best params:")
        display(best_trial.params)
        fig = plot_optimization_history(study)
        fig.update_layout(width=1000, height=450, margin=dict(r=180))
        fig.update_xaxes(range=[0, target_trials])
        display(fig)
    else:
        print("Best mean return: no complete trials yet")
